[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/01-python/07_tuples-sets.ipynb)

*Python from Scratch · Notebook 7 of 12*

# Tuples & Sets

Tuples: fixed records and unpacking. Sets: unique items, fast membership and set algebra.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A tuple is a sealed envelope; a set is a bag that keeps no duplicates and answers 'is it here?' instantly.

**What interviewers really test here:** Hashability, why set lookup is O(1) on average, and when to use each.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Tuples: lists that can't change *(Basics)*
- [ ] 2. Packing and unpacking *(Basics)*
- [ ] 3. Sets: unique items, fast lookup *(Core)*
- [ ] 4. Set operations *(Core)*
- [ ] 5. Hashability, frozenset and when to use what *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definitions

A **tuple** is an **ordered, immutable** sequence of items of any type, written `(a, b, c)`. It supports indexing, slicing, iteration, `len`, `in`, `count` and `index`, but no methods that change it.



A **set** is an **unordered, mutable** collection of **unique, hashable** items, written `{a, b, c}`. It is modelled on mathematical sets and implemented with a hash table.



A **frozenset** is an immutable (and therefore hashable) set.

### Tuple features

- **Creation**: `(1, 2)`, `1, 2`, `tuple([1, 2])`, `()`, and `(x,)` for one item.
- **Packing/unpacking**: `a, b = 1, 2`; extended unpacking `head, *tail = seq`.
- **Immutability is shallow**: `t = ([1], 2)` — you can't replace `t[0]`, but you can append to the list inside it. Such a tuple isn't hashable.
- **Named tuples**: `collections.namedtuple("Point", "x y")` gives tuples whose fields have names: `p.x`. `typing.NamedTuple` and dataclasses are modern alternatives.



#### Why use a tuple instead of a list?




- It signals "this group of values is fixed".
- It's hashable (if its items are), so it can be a dict key or set item.
- It's slightly smaller and faster to create than a list.

### Set features and operations

| Method | Effect | Time |
| --- | --- | --- |
| `add(x)` | insert x | O(1) |
| `remove(x)` | delete x, KeyError if missing | O(1) |
| `discard(x)` | delete x if present | O(1) |
| `pop()` | remove an arbitrary item | O(1) |
| `x in s` | membership | O(1) average |
| `update(it)` / `\|=` | add many | O(k) |
| `a \| b`, `a & b`, `a - b`, `a ^ b` | union, intersection, difference, symmetric difference | O(len a + len b) (intersection O(min)) |
| `a <= b`, `a < b` | subset, proper subset | O(len a) |



Set comprehension: `{x % 3 for x in nums}`. Note `{}` is an empty **dict**; an empty set is `set()`.

### Hashing and hashability

A **hash** is an integer computed from a value (`hash("abc")`). Sets and dicts use it to jump straight to the bucket where the item would be, which is why membership is O(1) on average.



An object is **hashable** if its hash never changes during its lifetime and it can be compared with `==`. Rule: equal objects must have equal hashes.



| Hashable | Not hashable |
| --- | --- |
| int, float, str, bool, None, tuple (of hashables), frozenset | list, dict, set, bytearray |

### Comparison: list, tuple, set

|   | list | tuple | set |
| --- | --- | --- | --- |
| Syntax | `[1, 2]` | `(1, 2)` | `{1, 2}` |
| Ordered | yes | yes | no |
| Duplicates | allowed | allowed | removed |
| Mutable | yes | no | yes |
| Indexing | yes | yes | no |
| Membership | O(n) | O(n) | O(1) |
| Hashable | no | yes\* | no (frozenset yes) |

### Applications

- **Tuples**: returning several values, coordinates, RGB colours, rows from a database, dictionary keys like `(row, col)`.
- **Sets**: removing duplicates, fast membership tests (visited nodes, banned words), comparing groups (common friends, missing items), tags.

---
# Part 2: Lessons, step by step

## Lesson 1: Tuples: lists that can't change

*Level: Basics*

A **tuple** is an ordered collection like a list, written with round brackets — but it is **immutable**: once created, you can't add, remove or replace items.



```python
point = (3, 4)
rgb = (255, 128, 0)
point[0]          # 3 — indexing and slicing work
point[0] = 9      # TypeError: 'tuple' object does not support item assignment
```



The comma makes the tuple, not the brackets: `t = 1, 2` is a tuple, and a one-item tuple needs a trailing comma: `(5,)`. `(5)` is just the number 5.



Use a tuple for a fixed group of values that belong together: coordinates, a date (y, m, d), a database row.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def tuple_demo():
    point = (3, 4)
    x = point[0]
    single = (5,)
    not_tuple = (5)
    t = 1, 2, 3
    print(point, x, single, not_tuple, t, type(single), type(not_tuple))


tuple_demo()

(3, 4) 3 (5,) 5 (1, 2, 3) <class 'tuple'> <class 'int'>


**✏️ Your turn:** you now have what you need for [Q5](#q5), [Q1](#q1). Try one now, or carry on and do them in Part 3.

## Lesson 2: Packing and unpacking

*Level: Basics*

**Packing**: writing values separated by commas creates a tuple. **Unpacking**: assigning a tuple to several names splits it:



```python
record = ("Ana", 21, "Pune")       # packing
name, age, city = record           # unpacking
a, b = b, a                        # swap: pack (b, a), then unpack
first, *rest = [1, 2, 3, 4]        # first = 1, rest = [2, 3, 4]
for name, score in [("a", 1), ("b", 2)]:   # unpacking in loops
    ...
```



Functions return several values the same way: `return lo, hi` returns one tuple, which the caller can unpack.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def unpack_demo():
    record = ("Ana", 21, "Pune")
    name, age, city = record
    first, *rest = [10, 20, 30, 40]

    def low_high(nums):
        return min(nums), max(nums)

    lo, hi = low_high([4, 9, 1])
    print(name, age, city, first, rest, lo, hi)


unpack_demo()

Ana 21 Pune 10 [20, 30, 40] 1 9


**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q2](#q2), [Q7](#q7), [Q16](#q16). Try one now, or carry on and do them in Part 3.

## Lesson 3: Sets: unique items, fast lookup

*Level: Core*

A **set** is an **unordered** collection of **unique** items, written with curly braces:



```python
colors = {"red", "green", "red"}   # {'red', 'green'} — duplicate dropped
empty = set()                      # {} would be an empty DICT
colors.add("blue")
colors.discard("pink")             # no error if missing (remove() would raise)
"red" in colors                    # True — and very fast
```



Sets have no index (`colors[0]` is an error) and no fixed order. In return, checking membership takes about the same time no matter how big the set is — O(1) — while a list must scan every item.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def set_basics_demo():
    nums = [3, 1, 3, 2, 1, 3]
    unique = set(nums)
    unique.add(10)
    unique.discard(99)
    has_two = 2 in unique
    print(unique, len(unique), has_two)


set_basics_demo()

{10, 1, 2, 3} 4 True


**✏️ Your turn:** you now have what you need for [Q8](#q8), [Q12](#q12), [Q14](#q14). Try one now, or carry on and do them in Part 3.

## Lesson 4: Set operations

*Level: Core*

| Operation | Operator | Method | {1, 2, 3} with {2, 3, 4} |
| --- | --- | --- | --- |
| Union (either) | `a \| b` | `a.union(b)` | {1, 2, 3, 4} |
| Intersection (both) | `a & b` | `a.intersection(b)` | {2, 3} |
| Difference (a only) | `a - b` | `a.difference(b)` | {1} |
| Symmetric difference (exactly one) | `a ^ b` | `a.symmetric_difference(b)` | {1, 4} |
| Subset / superset | `a <= b`, `a >= b` | `issubset`, `issuperset` | False |
| No common items? | — | `a.isdisjoint(b)` | False |



These are the Venn diagrams from maths class, built in.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def set_ops_demo():
    python_club = {"ana", "ben", "cy"}
    chess_club = {"ben", "dia"}
    both = python_club & chess_club
    either = python_club | chess_club
    only_python = python_club - chess_club
    exactly_one = python_club ^ chess_club
    print(both, either, only_python, exactly_one)


set_ops_demo()

{'ben'} {'ben', 'dia', 'ana', 'cy'} {'ana', 'cy'} {'ana', 'dia', 'cy'}


**✏️ Your turn:** you now have what you need for [Q9](#q9), [Q10](#q10), [Q11](#q11), [Q18](#q18), [Q13](#q13), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 5: Hashability, frozenset and when to use what

*Level: Advanced*

Sets and dict keys use **hashing**, so their items must be **hashable** — values that never change: numbers, strings, tuples of hashables, frozensets. Lists, dicts and sets are not hashable.



```python
visited = {(0, 0), (1, 2)}        # tuples are fine
bad = {[0, 0]}                    # TypeError: unhashable type: 'list'
frozen = frozenset({1, 2})        # an immutable set — can go inside a set or be a dict key
```



| Need | Use |
| --- | --- |
| ordered, changeable collection | list |
| fixed record, dict key, return several values | tuple |
| unique items, fast "is x in it?", set algebra | set |
| a set that must not change | frozenset |

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def hashable_demo():
    seen = set()
    for move in [(0, 1), (1, 1), (0, 1)]:
        seen.add(move)
    groups = {frozenset({"a", "b"}): "team 1"}
    print(seen, groups[frozenset({"b", "a"})])


hashable_demo()

{(0, 1), (1, 1)} team 1


**✏️ Your turn:** you now have what you need for [Q17](#q17), [Q20](#q20), [Q6](#q6). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Swap a Pair

🟢 Warm-up · *Tuple unpacking*

Given a pair `(a, b)`, return `(b, a)`.

**Examples**

- `pair = (1, 2)` → `(2, 1)`
- `pair = ('x', 'y')` → `('y', 'x')`

In [ ]:
def swap_pair(pair):
    """
    Given a pair (a, b), return (b, a).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(swap_pair, [
    (((1, 2),), (2, 1)),
    ((('x', 'y'),), ('y', 'x')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Read each item by index and build a new tuple.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Unpack into two names and pack them back the other way round.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Read each item by index and build a new tuple.

```python
def swap_pair(pair):
    first = pair[0]
    second = pair[1]
    return (second, first)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Unpack into two names and pack them back the other way round.

```python
def swap_pair(pair):
    a, b = pair
    return b, a
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Read each item by index and build a new tuple. That costs O(1) time and O(1) space."

"I can do better: Unpack into two names and pack them back the other way round. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/swap-a-pair)

<a id="q2"></a>
### Q2. Min and Max Together

🟢 Warm-up · *Returning a tuple*

Return `(smallest, largest)` of a non-empty list.

**Examples**

- `nums = [4, 1, 9, 3]` → `(1, 9)`
- `nums = [5]` → `(5, 5)`

In [ ]:
def min_max(nums):
    """
    Return (smallest, largest) of a non-empty list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(min_max, [
    (([4, 1, 9, 3],), (1, 9)),
    (([5],), (5, 5)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

One loop tracking both.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Return `min(nums), max(nums)` — a tuple is created by the comma.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** One loop tracking both.

```python
def min_max(nums):
    lo = hi = nums[0]
    for x in nums:
        if x < lo:
            lo = x
        if x > hi:
            hi = x
    return (lo, hi)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Return `min(nums), max(nums)` — a tuple is created by the comma.

```python
def min_max(nums):
    return min(nums), max(nums)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: One loop tracking both. That costs O(n) time and O(1) space."

"I can do better: Return `min(nums), max(nums)` — a tuple is created by the comma. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/min-and-max-together)

<a id="q3"></a>
### Q3. Describe a Record

🟢 Warm-up · *Unpacking*

A record is `(name, age, city)`. Return `'NAME (AGE) from CITY'`.

**Examples**

- `record = ('Ana', 21, 'Pune')` → `'Ana (21) from Pune'`
- `record = ('Li', 30, 'Oslo')` → `'Li (30) from Oslo'`

In [ ]:
def describe(record):
    """
    A record is (name, age, city). Return 'NAME (AGE) from CITY'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(describe, [
    ((('Ana', 21, 'Pune'),), 'Ana (21) from Pune'),
    ((('Li', 30, 'Oslo'),), 'Li (30) from Oslo'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Index each field: record[0], record[1], record[2].

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Unpack into meaningful names first — much easier to read.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Index each field: record[0], record[1], record[2].

```python
def describe(record):
    return record[0] + " (" + str(record[1]) + ") from " + record[2]
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Unpack into meaningful names first — much easier to read.

```python
def describe(record):
    name, age, city = record
    return f"{name} ({age}) from {city}"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Index each field: record[0], record[1], record[2]. That costs O(1) time and O(1) space."

"I can do better: Unpack into meaningful names first — much easier to read. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/describe-a-record)

<a id="q4"></a>
### Q4. Distance From Origin

🟢 Warm-up · *Tuple unpacking · math*

Return the distance of point `(x, y)` from (0, 0), rounded to 2 decimals.

**Examples**

- `point = (3, 4)` → `5.0`
- `point = (1, 1)` → `1.41`
- `point = (0, 0)` → `0.0`

In [ ]:
def from_origin(point):
    """
    Return the distance of point (x, y) from (0, 0), rounded to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(from_origin, [
    (((3, 4),), 5.0),
    (((1, 1),), 1.41),
    (((0, 0),), 0.0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Index the tuple and apply Pythagoras.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Unpack the tuple straight into `math.hypot(*point)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Index the tuple and apply Pythagoras.

```python
def from_origin(point):
    x = point[0]
    y = point[1]
    return round((x * x + y * y) ** 0.5, 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Unpack the tuple straight into `math.hypot(*point)`.

```python
def from_origin(point):
    import math
    return round(math.hypot(*point), 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Index the tuple and apply Pythagoras. That costs O(1) time and O(1) space."

"I can do better: Unpack the tuple straight into `math.hypot(*point)`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/distance-from-origin)

<a id="q5"></a>
### Q5. Count in a Tuple

🟢 Warm-up · *tuple.count*

Return how many times `value` appears in the tuple.

**Examples**

- `t = (1, 2, 1, 1), value = 1` → `3`
- `t = ('a',), value = 'b'` → `0`

In [ ]:
def tuple_count(t, value):
    """
    Return how many times value appears in the tuple.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(tuple_count, [
    (((1, 2, 1, 1), 1), 3),
    ((('a',), 'b'), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and count.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Tuples have `count` and `index` methods.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop and count.

```python
def tuple_count(t, value):
    count = 0
    for x in t:
        if x == value:
            count += 1
    return count
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Tuples have `count` and `index` methods.

```python
def tuple_count(t, value):
    return t.count(value)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and count. That costs O(n) time and O(1) space."

"I can do better: Tuples have `count` and `index` methods. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/count-in-a-tuple)

<a id="q6"></a>
### Q6. Pair Up Two Lists

🟢 Warm-up · *zip*

Return a list of pairs `(a[i], b[i])` for lists of equal length.

**Examples**

- `a = ['x', 'y'], b = [1, 2]` → `[('x', 1), ('y', 2)]`
- `a = [], b = []` → `[]`

In [ ]:
def pair_up(a, b):
    """
    Return a list of pairs (a[i], b[i]) for lists of equal length.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pair_up, [
    ((['x', 'y'], [1, 2]), [('x', 1), ('y', 2)]),
    (([], []), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over indices and build each tuple.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`list(zip(a, b))`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop over indices and build each tuple.

```python
def pair_up(a, b):
    pairs = []
    for i in range(len(a)):
        pairs.append((a[i], b[i]))
    return pairs
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `list(zip(a, b))`.

```python
def pair_up(a, b):
    return list(zip(a, b))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over indices and build each tuple. That costs O(n) time and O(n) space."

"I can do better: `list(zip(a, b))`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/pair-up-two-lists)

<a id="q7"></a>
### Q7. Sum, Mean and Max

🟢 Warm-up · *Returning several values*

Return `(total, mean, largest)` of a non-empty list; mean rounded to 2 decimals.

**Examples**

- `nums = [2, 4, 9]` → `(15, 5.0, 9)`
- `nums = [7]` → `(7, 7.0, 7)`

In [ ]:
def stats(nums):
    """
    Return (total, mean, largest) of a non-empty list; mean rounded to 2
    decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(stats, [
    (([2, 4, 9],), (15, 5.0, 9)),
    (([7],), (7, 7.0, 7)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

One loop computing everything by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Use the built-ins and return a tuple.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** One loop computing everything by hand.

```python
def stats(nums):
    total, largest = 0, nums[0]
    for x in nums:
        total += x
        if x > largest:
            largest = x
    return (total, round(total / len(nums), 2), largest)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Use the built-ins and return a tuple.

```python
def stats(nums):
    total = sum(nums)
    return total, round(total / len(nums), 2), max(nums)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: One loop computing everything by hand. That costs O(n) time and O(1) space."

"I can do better: Use the built-ins and return a tuple. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/sum-mean-and-max)

<a id="q8"></a>
### Q8. Count Unique Values

🟢 Warm-up · *set()*

Return how many different values are in the list.

**Examples**

- `items = [1, 2, 2, 3, 3, 3]` → `3`
- `items = []` → `0`
- `items = ['a', 'A']` → `2`

In [ ]:
def unique_count(items):
    """
    Return how many different values are in the list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(unique_count, [
    (([1, 2, 2, 3, 3, 3],), 3),
    (([],), 0),
    ((['a', 'A'],), 2),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build a list of uniques, checking `in` each time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set keeps only unique values: `len(set(items))`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Build a list of uniques, checking `in` each time.

```python
def unique_count(items):
    seen = []
    for x in items:
        if x not in seen:
            seen.append(x)
    return len(seen)
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A set keeps only unique values: `len(set(items))`.

```python
def unique_count(items):
    return len(set(items))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build a list of uniques, checking `in` each time. That costs O(n²) time and O(n) space."

"I can do better: A set keeps only unique values: `len(set(items))`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/count-unique-values)

<a id="q9"></a>
### Q9. Common Letters

🟢 Warm-up · *Intersection*

Return the sorted list of letters that appear in both words.

**Examples**

- `a = 'python', b = 'typhoon'` → `['h', 'n', 'o', 'p', 't', 'y']`
- `a = 'abc', b = 'xyz'` → `[]`

In [ ]:
def common_letters(a, b):
    """
    Return the sorted list of letters that appear in both words.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(common_letters, [
    (('python', 'typhoon'), ['h', 'n', 'o', 'p', 't', 'y']),
    (('abc', 'xyz'), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each letter of the first word, scan the second; avoid duplicates.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Intersect two sets: `set(a) & set(b)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** For each letter of the first word, scan the second; avoid duplicates.

```python
def common_letters(a, b):
    result = []
    for ch in a:
        if ch in b and ch not in result:
            result.append(ch)
    return sorted(result)
```

**Cost:** O(m · n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Intersect two sets: `set(a) & set(b)`.

```python
def common_letters(a, b):
    return sorted(set(a) & set(b))
```

**Cost:** O(m + n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each letter of the first word, scan the second; avoid duplicates. That costs O(m · n) time and O(k) space."

"I can do better: Intersect two sets: `set(a) & set(b)`. That's O(m + n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/common-letters)

<a id="q10"></a>
### Q10. Union of Two Lists

🟢 Warm-up · *Union*

Return all values that appear in either list, sorted, without duplicates.

**Examples**

- `a = [3, 1, 2], b = [2, 5]` → `[1, 2, 3, 5]`
- `a = [], b = [1, 1]` → `[1]`

In [ ]:
def union(a, b):
    """
    Return all values that appear in either list, sorted, without duplicates.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(union, [
    (([3, 1, 2], [2, 5]), [1, 2, 3, 5]),
    (([], [1, 1]), [1]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add both lists into a result, skipping values already present.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`sorted(set(a) | set(b))`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Add both lists into a result, skipping values already present.

```python
def union(a, b):
    result = []
    for x in a + b:
        if x not in result:
            result.append(x)
    return sorted(result)
```

**Cost:** O((m + n)²) time, O(m + n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `sorted(set(a) | set(b))`.

```python
def union(a, b):
    return sorted(set(a) | set(b))
```

**Cost:** O((m + n) log) time, O(m + n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add both lists into a result, skipping values already present. That costs O((m + n)²) time and O(m + n) space."

"I can do better: `sorted(set(a) | set(b))`. That's O((m + n) log) time and O(m + n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/union-of-two-lists)

<a id="q11"></a>
### Q11. In First List Only

🟢 Warm-up · *Difference*

Return the sorted unique values that are in `a` but not in `b`.

**Examples**

- `a = [1, 2, 3, 4], b = [2, 4]` → `[1, 3]`
- `a = [1, 1], b = [1]` → `[]`

In [ ]:
def only_in_first(a, b):
    """
    Return the sorted unique values that are in a but not in b.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(only_in_first, [
    (([1, 2, 3, 4], [2, 4]), [1, 3]),
    (([1, 1], [1]), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each value of a, scan b.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`set(a) - set(b)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** For each value of a, scan b.

```python
def only_in_first(a, b):
    result = []
    for x in a:
        if x not in b and x not in result:
            result.append(x)
    return sorted(result)
```

**Cost:** O(m · n) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `set(a) - set(b)`.

```python
def only_in_first(a, b):
    return sorted(set(a) - set(b))
```

**Cost:** O(m + n) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each value of a, scan b. That costs O(m · n) time and O(m) space."

"I can do better: `set(a) - set(b)`. That's O(m + n) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/in-first-list-only)

<a id="q12"></a>
### Q12. Has Duplicates?

🟢 Warm-up · *set size*

Return True if any value appears more than once.

**Examples**

- `items = [1, 2, 3, 1]` → `True`
- `items = [1, 2, 3]` → `False`
- `items = []` → `False`

In [ ]:
def has_duplicates(items):
    """
    Return True if any value appears more than once.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(has_duplicates, [
    (([1, 2, 3, 1],), True),
    (([1, 2, 3],), False),
    (([],), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare every pair of items.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set drops duplicates, so compare sizes.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Compare every pair of items.

```python
def has_duplicates(items):
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            if items[i] == items[j]:
                return True
    return False
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A set drops duplicates, so compare sizes.

```python
def has_duplicates(items):
    return len(set(items)) != len(items)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare every pair of items. That costs O(n²) time and O(1) space."

"I can do better: A set drops duplicates, so compare sizes. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/has-duplicates)

<a id="q13"></a>
### Q13. Is It a Subset?

🟢 Warm-up · *Subset test*

Return True if every value of `small` also appears in `big`.

**Examples**

- `small = [1, 2], big = [3, 2, 1]` → `True`
- `small = [1, 4], big = [1, 2, 3]` → `False`
- `small = [], big = [1]` → `True`

In [ ]:
def is_subset(small, big):
    """
    Return True if every value of small also appears in big.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_subset, [
    (([1, 2], [3, 2, 1]), True),
    (([1, 4], [1, 2, 3]), False),
    (([], [1]), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check each value with `in` on the list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`set(small) <= set(big)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check each value with `in` on the list.

```python
def is_subset(small, big):
    for x in small:
        if x not in big:
            return False
    return True
```

**Cost:** O(m · n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `set(small) <= set(big)`.

```python
def is_subset(small, big):
    return set(small) <= set(big)
```

**Cost:** O(m + n) time, O(m + n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check each value with `in` on the list. That costs O(m · n) time and O(1) space."

"I can do better: `set(small) <= set(big)`. That's O(m + n) time and O(m + n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/is-it-a-subset)

<a id="q14"></a>
### Q14. Unique Words, Ignoring Case

🟢 Warm-up · *Set comprehension*

Return how many different words are in the sentence, ignoring upper/lower case.

**Examples**

- `sentence = 'The cat and the hat'` → `4`
- `sentence = 'a A a'` → `1`

In [ ]:
def unique_words(sentence):
    """
    Return how many different words are in the sentence, ignoring upper/lower
    case.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(unique_words, [
    (('The cat and the hat',), 4),
    (('a A a',), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep a list of lower-cased words seen so far.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set comprehension of lower-cased words.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Keep a list of lower-cased words seen so far.

```python
def unique_words(sentence):
    seen = []
    for w in sentence.split():
        w = w.lower()
        if w not in seen:
            seen.append(w)
    return len(seen)
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A set comprehension of lower-cased words.

```python
def unique_words(sentence):
    return len({w.lower() for w in sentence.split()})
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep a list of lower-cased words seen so far. That costs O(n²) time and O(n) space."

"I can do better: A set comprehension of lower-cased words. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/unique-words-ignoring-case)

<a id="q15"></a>
### Q15. Sort Pairs by Second Item

🟡 Interview standard · *sorted with key*

Sort a list of (name, score) pairs by score, lowest first.

**Examples**

- `pairs = [('a', 3), ('b', 1), ('c', 2)]` → `[('b', 1), ('c', 2), ('a', 3)]`
- `pairs = []` → `[]`

In [ ]:
def sort_by_score(pairs):
    """
    Sort a list of (name, score) pairs by score, lowest first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sort_by_score, [
    (([('a', 3), ('b', 1), ('c', 2)],), [('b', 1), ('c', 2), ('a', 3)]),
    (([],), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Selection sort by hand comparing the second items.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`sorted(pairs, key=lambda p: p[1])`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Selection sort by hand comparing the second items.

```python
def sort_by_score(pairs):
    items = pairs[:]
    for i in range(len(items)):
        smallest = i
        for j in range(i + 1, len(items)):
            if items[j][1] < items[smallest][1]:
                smallest = j
        items[i], items[smallest] = items[smallest], items[i]
    return items
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `sorted(pairs, key=lambda p: p[1])`.

```python
def sort_by_score(pairs):
    return sorted(pairs, key=lambda p: p[1])
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Selection sort by hand comparing the second items. That costs O(n²) time and O(n) space."

"I can do better: `sorted(pairs, key=lambda p: p[1])`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/sort-pairs-by-second-item)

<a id="q16"></a>
### Q16. Unzip Pairs

🟡 Interview standard · *zip(\*pairs)*

Turn a list of pairs into two tuples: `[(1, 'a'), (2, 'b')] → ((1, 2), ('a', 'b'))`.

**Examples**

- `pairs = [(1, 'a'), (2, 'b')]` → `((1, 2), ('a', 'b'))`
- `pairs = [(5, 6)]` → `((5,), (6,))`

In [ ]:
def unzip(pairs):
    """
    Turn a list of pairs into two tuples: [(1, 'a'), (2, 'b')] → ((1, 2), ('a',
    'b')).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(unzip, [
    (([(1, 'a'), (2, 'b')],), ((1, 2), ('a', 'b'))),
    (([(5, 6)],), ((5,), (6,))),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two loops collecting firsts and seconds, then convert to tuples.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`zip(*pairs)` passes every pair as a separate argument, regrouping them by position.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Two loops collecting firsts and seconds, then convert to tuples.

```python
def unzip(pairs):
    firsts, seconds = [], []
    for p in pairs:
        firsts.append(p[0])
        seconds.append(p[1])
    return tuple(firsts), tuple(seconds)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `zip(*pairs)` passes every pair as a separate argument, regrouping them by position.

```python
def unzip(pairs):
    firsts, seconds = zip(*pairs)
    return firsts, seconds
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two loops collecting firsts and seconds, then convert to tuples. That costs O(n) time and O(n) space."

"I can do better: `zip(*pairs)` passes every pair as a separate argument, regrouping them by position. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/unzip-pairs)

<a id="q17"></a>
### Q17. Distinct Cells Visited

🟡 Interview standard · *Tuples in a set*

A robot starts at (0, 0) and follows moves 'U', 'D', 'L', 'R'. Return how many different cells it visits (including the start).

**Examples**

- `moves = 'URDL'` → `4`
- `moves = 'RRRR'` → `5`
- `moves = ''` → `1`

In [ ]:
def cells_visited(moves):
    """
    A robot starts at (0, 0) and follows moves 'U', 'D', 'L', 'R'. Return how
    many different cells it visits (including the start).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cells_visited, [
    (('URDL',), 4),
    (('RRRR',), 5),
    (('',), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep a list of visited cells and check `in` before adding (each check scans the list).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Store the (x, y) tuples in a set — tuples are hashable, and duplicates disappear.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Keep a list of visited cells and check `in` before adding (each check scans the list).

```python
def cells_visited(moves):
    x, y = 0, 0
    visited = [(0, 0)]
    for m in moves:
        if m == "U":
            y += 1
        elif m == "D":
            y -= 1
        elif m == "L":
            x -= 1
        else:
            x += 1
        if (x, y) not in visited:
            visited.append((x, y))
    return len(visited)
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Store the (x, y) tuples in a set — tuples are hashable, and duplicates disappear.

```python
def cells_visited(moves):
    step = {"U": (0, 1), "D": (0, -1), "L": (-1, 0), "R": (1, 0)}
    x = y = 0
    seen = {(0, 0)}
    for m in moves:
        dx, dy = step[m]
        x, y = x + dx, y + dy
        seen.add((x, y))
    return len(seen)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep a list of visited cells and check `in` before adding (each check scans the list). That costs O(n²) time and O(n) space."

"I can do better: Store the (x, y) tuples in a set — tuples are hashable, and duplicates disappear. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/distinct-cells-visited)

<a id="q18"></a>
### Q18. In Exactly One List

🟡 Interview standard · *Symmetric difference*

Return the sorted values that appear in exactly one of the two lists.

**Examples**

- `a = [1, 2, 3], b = [2, 3, 4]` → `[1, 4]`
- `a = [1], b = [1]` → `[]`

In [ ]:
def exactly_one_list(a, b):
    """
    Return the sorted values that appear in exactly one of the two lists.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(exactly_one_list, [
    (([1, 2, 3], [2, 3, 4]), [1, 4]),
    (([1], [1]), []),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check each value of a against b and each value of b against a.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`set(a) ^ set(b)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check each value of a against b and each value of b against a.

```python
def exactly_one_list(a, b):
    result = []
    for x in a:
        if x not in b and x not in result:
            result.append(x)
    for x in b:
        if x not in a and x not in result:
            result.append(x)
    return sorted(result)
```

**Cost:** O(m · n) time, O(m + n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `set(a) ^ set(b)`.

```python
def exactly_one_list(a, b):
    return sorted(set(a) ^ set(b))
```

**Cost:** O(m + n) time, O(m + n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check each value of a against b and each value of b against a. That costs O(m · n) time and O(m + n) space."

"I can do better: `set(a) ^ set(b)`. That's O(m + n) time and O(m + n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/in-exactly-one-list)

<a id="q19"></a>
### Q19. Missing Numbers

🟡 Interview standard · *Difference with a range*

`nums` should contain every number from 1 to n. Return the missing ones, sorted.

**Examples**

- `nums = [4, 1, 2, 7], n = 7` → `[3, 5, 6]`
- `nums = [1, 2, 3], n = 3` → `[]`
- `nums = [], n = 2` → `[1, 2]`

In [ ]:
def missing(nums, n):
    """
    nums should contain every number from 1 to n. Return the missing ones,
    sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(missing, [
    (([4, 1, 2, 7], 7), [3, 5, 6]),
    (([1, 2, 3], 3), []),
    (([], 2), [1, 2]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each number 1..n, search the list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`set(range(1, n + 1)) - set(nums)`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** For each number 1..n, search the list.

```python
def missing(nums, n):
    result = []
    for k in range(1, n + 1):
        if k not in nums:
            result.append(k)
    return result
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `set(range(1, n + 1)) - set(nums)`.

```python
def missing(nums, n):
    return sorted(set(range(1, n + 1)) - set(nums))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each number 1..n, search the list. That costs O(n²) time and O(n) space."

"I can do better: `set(range(1, n + 1)) - set(nums)`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/missing-numbers)

<a id="q20"></a>
### Q20. Two Numbers Add Up

🟡 Interview standard · *Set of seen values*

Return True if two different positions in the list add up to `target`.

**Examples**

- `nums = [3, 8, 1, 5], target = 9` → `True`
- `nums = [1, 2, 3], target = 10` → `False`
- `nums = [4], target = 8` → `False`

In [ ]:
def two_add_up(nums, target):
    """
    Return True if two different positions in the list add up to target.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(two_add_up, [
    (([3, 8, 1, 5], 9), True),
    (([1, 2, 3], 10), False),
    (([4], 8), False),
    (([4, 4], 8), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check every pair with nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Walk once; for each x ask whether target − x was already seen.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check every pair with nested loops.

```python
def two_add_up(nums, target):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return True
    return False
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Walk once; for each x ask whether target − x was already seen.

```python
def two_add_up(nums, target):
    seen = set()
    for x in nums:
        if target - x in seen:
            return True
        seen.add(x)
    return False
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check every pair with nested loops. That costs O(n²) time and O(1) space."

"I can do better: Walk once; for each x ask whether target − x was already seen. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/tuples-sets/two-numbers-add-up)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What's the difference between a list and a tuple?</b></summary>

Lists are mutable (items can change) and tuples are immutable. Tuples can be hashed and used as dict keys; lists can't. Use tuples for fixed records and lists for collections that change.

</details>

<details>
<summary><b>2. How do you create a tuple with one element?</b></summary>

Add a trailing comma: `(5,)` or `5,`. `(5)` is just the int 5 in brackets.

</details>

<details>
<summary><b>3. Can a tuple contain a list? Is it still immutable?</b></summary>

Yes. The tuple can't change which objects it holds, but the list inside can still be modified. Such a tuple is not hashable.

</details>

<details>
<summary><b>4. Why is {} not an empty set?</b></summary>

Curly braces were used for dicts first, so `{}` is an empty dict. Create an empty set with `set()`.

</details>

<details>
<summary><b>5. remove() vs discard() for sets?</b></summary>

Both delete an item. `remove` raises KeyError if the item is missing; `discard` does nothing.

</details>

<details>
<summary><b>6. Why is membership faster in a set than in a list?</b></summary>

A set hashes the value and checks one bucket (O(1) average). A list compares the value with each element in order (O(n)).

</details>

<details>
<summary><b>7. What is a frozenset?</b></summary>

An immutable set. It's hashable, so it can be stored in another set or used as a dict key.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Tuples: lists that can't change
- [ ] I can explain: Packing and unpacking
- [ ] I can explain: Sets: unique items, fast lookup
- [ ] I can explain: Set operations
- [ ] I can explain: Hashability, frozenset and when to use what

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Unzip Pairs
- [ ] [Q17](#q17) Distinct Cells Visited
- [ ] [Q18](#q18) In Exactly One List
- [ ] [Q19](#q19) Missing Numbers
- [ ] [Q20](#q20) Two Numbers Add Up

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.